# Qwen Majority Vote vs Manual Annotations

This notebook aggregates Qwen outputs across multiple prompt strategies (for example `direct`, `expert`, `inverse`) and uses a **majority vote** as the final label per image. It then compares that voted label to manual annotations and reports confusion matrix, accuracy, and class-wise metrics.

In [15]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [16]:
# ===== PATH CONFIGURATION =====
BASE_DIR = Path('/mnt/lustre/work/hein/hof217/stable_diffusion')
NEW_QWEN_BASE = BASE_DIR / 'x7-z1-qwen_small/new_qwen_outputs'
MANUAL_DIR = BASE_DIR / 'z5-manual_annotations'

MODELS = ['standard', 'cogvlm', 'redcaps']
MANUAL_FILES = {
    'standard': 'standard.csv',
    'cogvlm': 'cogvlm_clic.csv',
    'redcaps': 'redcaps_clic.csv',
}

LABELS = ['Yes', 'No', 'Ambiguous']

In [17]:
def parse_manual_indices(s):
    if pd.isna(s) or str(s).strip() == '':
        return set()
    return set(int(x.strip()) for x in str(s).replace(';', ',').split(',') if x.strip().isdigit())


def normalize_qwen_answer(ans):
    if pd.isna(ans) or str(ans).strip() == '':
        return 'Ambiguous'
    v = str(ans).strip().lower().rstrip('.,!?;:')
    if v == 'yes':
        return 'Yes'
    if v == 'no':
        return 'No'
    return 'Ambiguous'


def get_qwen_answer(row):
    pred = row.get('predicted_label', np.nan)
    if pd.notna(pred) and str(pred).strip() != '':
        return normalize_qwen_answer(pred)
    return normalize_qwen_answer(row.get('full_output', np.nan))


def load_manual_for_phrase(model, phrase):
    manual_path = MANUAL_DIR / MANUAL_FILES[model]
    if not manual_path.exists():
        return None

    df = pd.read_csv(manual_path)
    row = df[df['phrase'] == phrase]
    if row.empty:
        return None

    row = row.iloc[0]
    if pd.isna(row.get('checked', np.nan)):
        return None

    return {
        'tensor_file': row['tensor_file'],
        'checked': int(row['checked']),
        'ambiguous': parse_manual_indices(row['[tensor_index-es for ambigious images]']),
        'incorrect': parse_manual_indices(row['[tensor_index-es for incorrect images]']),
    }


def manual_label(idx, manual_data):
    if manual_data is None or idx >= manual_data['checked']:
        return None
    if idx in manual_data['incorrect']:
        return 'No'
    if idx in manual_data['ambiguous']:
        return 'Ambiguous'
    return 'Yes'


def load_qwen_strategy(model, phrase, strategy):
    phrase_file_map = {'close by': 'closeby'}
    phrase_file = phrase_file_map.get(phrase, phrase.replace(' ', '_'))

    qwen_dir = NEW_QWEN_BASE / model / strategy
    files = sorted(qwen_dir.glob(f'qwen_{phrase_file}_*.csv'))
    if not files:
        return pd.DataFrame()

    frames = [pd.read_csv(f) for f in files]
    df = pd.concat(frames, ignore_index=True)
    if 'phrase' in df.columns:
        df = df[df['phrase'] == phrase]

    if df.empty:
        return pd.DataFrame()

    df = df.drop_duplicates(subset=['tensor_file', 'tensor_index'], keep='last').copy()
    df['qwen_short'] = df.apply(get_qwen_answer, axis=1)

    keep_cols = ['tensor_file', 'tensor_index', 'class1', 'class2', 'qwen_short']
    for col in ['class1', 'class2']:
        if col not in df.columns:
            df[col] = ''
    return df[keep_cols]


def create_confusion_matrix(y_true, y_pred, labels=LABELS):
    matrix = pd.DataFrame(0, index=labels, columns=labels)
    for t, p in zip(y_true, y_pred):
        if t in labels and p in labels:
            matrix.loc[t, p] += 1
    return matrix


def per_class_metrics(conf_matrix, labels=LABELS):
    rows = []
    for lab in labels:
        tp = conf_matrix.loc[lab, lab]
        fp = conf_matrix[lab].sum() - tp
        fn = conf_matrix.loc[lab].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        support = conf_matrix.loc[lab].sum()
        rows.append({
            'label': lab,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': int(support),
        })

    out = pd.DataFrame(rows)
    macro = {
        'label': 'macro_avg',
        'precision': out['precision'].mean(),
        'recall': out['recall'].mean(),
        'f1': out['f1'].mean(),
        'support': int(out['support'].sum()),
    }
    out = pd.concat([out, pd.DataFrame([macro])], ignore_index=True)
    return out


def build_majority_vote_table(model, phrase, strategies=('direct', 'expert', 'inverse')):
    strategy_tables = {}
    merged = None

    for strat in strategies:
        sdf = load_qwen_strategy(model, phrase, strat)
        strategy_tables[strat] = sdf

        if sdf.empty:
            continue

        label_col = sdf[['tensor_file', 'tensor_index', 'qwen_short']].rename(columns={'qwen_short': f'label_{strat}'})
        meta_col = sdf[['tensor_file', 'tensor_index', 'class1', 'class2']].copy()

        if merged is None:
            merged = pd.merge(label_col, meta_col, on=['tensor_file', 'tensor_index'], how='left')
        else:
            merged = pd.merge(merged, label_col, on=['tensor_file', 'tensor_index'], how='outer')
            merged = pd.merge(
                merged,
                meta_col,
                on=['tensor_file', 'tensor_index'],
                how='left',
                suffixes=('', '_new')
            )
            for c in ['class1', 'class2']:
                if f'{c}_new' in merged.columns:
                    merged[c] = merged[c].fillna(merged[f'{c}_new'])
                    merged.drop(columns=[f'{c}_new'], inplace=True)

    if merged is None or merged.empty:
        return pd.DataFrame(), strategy_tables

    label_cols = [f'label_{s}' for s in strategies if f'label_{s}' in merged.columns]

    def vote_row(row):
        votes = [row[c] for c in label_cols if pd.notna(row[c]) and str(row[c]).strip() != '']
        if not votes:
            return pd.Series({
                'vote_count': 0,
                'vote_tie': False,
                'majority_label': np.nan,
                'vote_detail': ''
            })

        counts = Counter(votes)
        max_count = max(counts.values())
        winners = sorted([k for k, v in counts.items() if v == max_count])

        tie = len(winners) > 1
        majority = 'Ambiguous' if tie else winners[0]
        detail = '; '.join([f'{k}:{counts[k]}' for k in sorted(counts.keys())])

        return pd.Series({
            'vote_count': len(votes),
            'vote_tie': tie,
            'majority_label': majority,
            'vote_detail': detail
        })

    merged[['vote_count', 'vote_tie', 'majority_label', 'vote_detail']] = merged.apply(vote_row, axis=1)
    merged['prompt'] = (merged['class1'].fillna('') + ' ' + phrase + ' ' + merged['class2'].fillna('')).str.strip()
    return merged, strategy_tables


def strategy_accuracy_table(manual_data, strategy_tables):
    rows = []
    for strat, sdf in strategy_tables.items():
        if sdf.empty:
            rows.append({'strategy': strat, 'n_compared': 0, 'accuracy': np.nan})
            continue

        tmp = sdf.copy()
        tmp['manual_label'] = tmp['tensor_index'].astype(int).apply(lambda idx: manual_label(idx, manual_data))
        tmp = tmp[tmp['manual_label'].notna()]

        if tmp.empty:
            rows.append({'strategy': strat, 'n_compared': 0, 'accuracy': np.nan})
            continue

        acc = (tmp['manual_label'] == tmp['qwen_short']).mean()
        rows.append({'strategy': strat, 'n_compared': int(len(tmp)), 'accuracy': float(acc)})

    return pd.DataFrame(rows)

In [18]:
# Batch export for all models and all phrases
TARGET_STRATEGIES = ['direct', 'expert', 'inverse']
OUTPUT_DIR = BASE_DIR / 'x7-z1-qwen_small'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_rows = []

for model in MODELS:
    manual_df = pd.read_csv(MANUAL_DIR / MANUAL_FILES[model])
    phrases = [p for p in manual_df['phrase'].dropna().unique().tolist() if str(p).strip() != '']

    for phrase in phrases:
        manual = load_manual_for_phrase(model, phrase)
        if manual is None:
            continue

        # Manual distribution from checked annotations metadata
        checked = int(manual['checked'])
        manual_no = len([x for x in manual['incorrect'] if x < checked])
        manual_ambi = len([x for x in manual['ambiguous'] if x < checked])
        manual_yes = max(checked - manual_no - manual_ambi, 0)
        manual_distribution = f"Yes:{manual_yes},No:{manual_no},Ambiguous:{manual_ambi}"

        vote_df, strategy_tables = build_majority_vote_table(model, phrase, TARGET_STRATEGIES)
        found_strategies = [s for s in TARGET_STRATEGIES if s in strategy_tables and not strategy_tables[s].empty]

        accuracy_pct = np.nan
        yes_count = 0
        no_count = 0
        ambi_count = 0

        if not vote_df.empty:
            vote_df['manual_label'] = vote_df['tensor_index'].astype(int).apply(lambda idx: manual_label(idx, manual))
            eval_df = vote_df[vote_df['manual_label'].notna()].copy()

            if not eval_df.empty:
                accuracy_ratio = (eval_df['manual_label'] == eval_df['majority_label']).mean()
                accuracy_pct = float(accuracy_ratio * 100.0)
                pred_counts = eval_df['majority_label'].value_counts()
                yes_count = int(pred_counts.get('Yes', 0))
                no_count = int(pred_counts.get('No', 0))
                ambi_count = int(pred_counts.get('Ambiguous', 0))

        majority_distribution = f"Yes:{yes_count},No:{no_count},Ambiguous:{ambi_count}"

        row = {
            'model': model,
            'phrase': phrase,
            'manually_annotated_count': checked,
            'manually_annotated_distribution': manual_distribution,
            'majority_voted_districution': majority_distribution,
            'accuracy': round(accuracy_pct, 2) if pd.notna(accuracy_pct) else np.nan,
            'strategies_found': ','.join(found_strategies),
        }
        all_rows.append(row)

all_out = pd.DataFrame(all_rows)
if not all_out.empty:
    all_out = all_out.sort_values(by=['model', 'phrase'], kind='stable')
    all_path = OUTPUT_DIR / 'majority_vote_accuracy_all_models.csv'
    all_out.to_csv(all_path, index=False)

    print('Created CSV file:')
    print(f'  - {all_path}')
    display(all_out)
else:
    print('No rows exported.')

Created CSV file:
  - /mnt/lustre/work/hein/hof217/stable_diffusion/x7-z1-qwen_small/majority_vote_accuracy_all_models.csv


,model,phrase,manually_annotated_count,manually_annotated_distribution,majority_voted_districution,accuracy,strategies_found
11,cogvlm,above,20,"Yes:16,No:4,Ambiguous:0","Yes:14,No:5,Ambiguous:1",90.00,"direct,expert,inverse"
16,cogvlm,behind,10,"Yes:5,No:5,Ambiguous:0","Yes:3,No:6,Ambiguous:1",60.00,"direct,expert,inverse"
12,cogvlm,below,20,"Yes:13,No:7,Ambiguous:0","Yes:11,No:9,Ambiguous:0",90.00,"direct,expert,inverse"
15,cogvlm,between,20,"Yes:17,No:3,Ambiguous:0","Yes:8,No:12,Ambiguous:0",55.00,"direct,expert,inverse"
17,cogvlm,closeby,10,"Yes:10,No:0,Ambiguous:0","Yes:8,No:1,Ambiguous:1",80.00,"direct,expert,inverse"
18,cogvlm,far from,10,"Yes:6,No:4,Ambiguous:0","Yes:0,No:9,Ambiguous:1",40.00,"direct,expert,inverse"
19,cogvlm,in front of,10,"Yes:9,No:1,Ambiguous:0","Yes:8,No:2,Ambiguous:0",70.00,"direct,expert,inverse"
20,cogvlm,on top of,10,"Yes:10,No:0,Ambiguous:0","Yes:2,No:5,Ambiguous:3",20.00,"direct,expert"
22,cogvlm,over,10,"Yes:9,No:1,Ambiguous:0","Yes:3,No:6,Ambiguous:1",40.00,"direct,expert,inverse"
13,cogvlm,to the left of,20,"Yes:14,No:6,Ambiguous:0","Yes:10,No:8,Ambiguous:2",75.00,"direct,expert,inverse"


In [12]:
# Optional export of evaluated rows for external analysis
# if manual is not None and 'eval_df' in locals() and not eval_df.empty:
#     out_path = BASE_DIR / 'z5-manual_annotations' / f"majority_vote_eval_{MODEL}_{PHRASE.replace(' ', '_')}.csv"
#     eval_df.to_csv(out_path, index=False)
#     print(f'Saved: {out_path}')